# 🚀 Delta Lake Performance Benchmark
## Partitioning vs Z-ORDER vs Liquid Clustering

Benchmark controlado para investigar como diferentes estratégias de organização física influenciam workloads analíticos no Delta Lake.

**Pergunta principal:** qual estratégia apresenta melhor comportamento para cada padrão de consulta?

Estratégias:
- Hive-style Partitioning
- Z-ORDER
- Liquid Clustering

Conceitos:
- Data Skipping / file pruning
- Small Files
- OPTIMIZE
- Scan e arquivos
- Shuffle e plano físico
- Warm-up e múltiplas execuções
- Mediana como métrica principal

> Os resultados são específicos do runtime/cluster utilizado. O objetivo é demonstrar metodologia de benchmark, não estabelecer ganhos universais.

## 1. Metodologia

1. Criar dataset determinístico.
2. Usar o mesmo conteúdo lógico nas três tabelas.
3. Aplicar estratégias diferentes.
4. Executar as mesmas queries.
5. Fazer warm-up.
6. Medir múltiplas execuções.
7. Comparar medianas.
8. Inspecionar `EXPLAIN FORMATTED`.
9. Coletar métricas físicas.
10. Concluir somente após as medições.

### Experimentos

**A — Agregação**

```sql
SELECT categoria, AVG(valor)
FROM tabela
GROUP BY categoria
```

**B — Filtro seletivo / Data Skipping**

```sql
SELECT COUNT(*)
FROM tabela
WHERE cliente_id = ?
  AND data_id BETWEEN ? AND ?
```

**C — Join**

```sql
SELECT c.segmento, SUM(v.valor)
FROM vendas v
JOIN clientes c ON v.cliente_id = c.cliente_id
GROUP BY c.segmento
```

In [0]:
import time
import statistics
from pyspark.sql import functions as F

DB = "liquid_clustering_benchmark"
N_VENDAS = 500_000
N_CLIENTES = 50_000
N_REPETICOES = 5
SEED = 42

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {DB}")
spark.sql(f"USE {DB}")

print(f"Schema: {DB}")
print(f"Vendas: {N_VENDAS:,}")
print(f"Clientes: {N_CLIENTES:,}")
print(f"Repetições: {N_REPETICOES}")

Schema: liquid_clustering_benchmark
Vendas: 500,000
Clientes: 50,000
Repetições: 5


## 2. Geração dos dados

Os dados são determinísticos para que as três estratégias recebam exatamente o mesmo conteúdo lógico.

In [0]:
for t in [
    "vendas_partitioned",
    "vendas_zorder",
    "vendas_liquid",
    "clientes",
    "vendas_small_files"
]:
    spark.sql(f"DROP TABLE IF EXISTS {t}")

clientes = (
    spark.range(0, N_CLIENTES)
    .select(
        F.col("id").alias("cliente_id"),
        (F.col("id") % 10).cast("int").alias("segmento_id"),
        F.concat(F.lit("SEG_"), (F.col("id") % 10)).alias("segmento")
    )
)
clientes.write.format("delta").mode("overwrite").saveAsTable("clientes")

vendas = (
    spark.range(0, N_VENDAS)
    .select(
        F.col("id").alias("venda_id"),
        (F.col("id") % N_CLIENTES).cast("long").alias("cliente_id"),
        (F.col("id") % 20).cast("int").alias("categoria"),
        (F.col("id") % 365).cast("int").alias("data_id"),
        (
            F.pmod(F.xxhash64(F.col("id"), F.lit(SEED)), F.lit(500000)) / 100.0
        ).cast("double").alias("valor")
    )
)

print("Vendas:", vendas.count())
print("Clientes:", clientes.count())

Vendas: 500000
Clientes: 50000


## 3. Construção das estratégias

### A. Partitioning
Particionamento por `categoria`, uma coluna de baixa cardinalidade.

### B. Z-ORDER
Tabela Delta sem partição fixa, seguida de `OPTIMIZE ... ZORDER BY`.

### C. Liquid Clustering
Tabela Delta com `CLUSTER BY (cliente_id, data_id)` seguida de `OPTIMIZE`.

In [0]:
(
    vendas.write
    .format("delta")
    .mode("overwrite")
    .partitionBy("categoria")
    .saveAsTable("vendas_partitioned")
)

(
    vendas.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("vendas_zorder")
)

(
    vendas.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("vendas_liquid")
)

spark.sql("""ALTER TABLE vendas_liquid
CLUSTER BY (cliente_id, data_id)""")

print("Tabelas criadas.")

Tabelas criadas.


## 4. Otimização física

Aplicamos as operações de manutenção antes do benchmark.

In [0]:
spark.sql("""OPTIMIZE vendas_zorder
ZORDER BY (cliente_id, data_id)""")

spark.sql("OPTIMIZE vendas_liquid")

print("OPTIMIZE concluído.")

OPTIMIZE concluído.


## 5. Métricas físicas

`DESCRIBE DETAIL` permite observar tamanho, quantidade de arquivos, particionamento e clustering.

In [0]:
def table_detail(table_name):
    return (
        spark.sql(f"DESCRIBE DETAIL {table_name}")
        .select(
            "name",
            "numFiles",
            "sizeInBytes",
            "format",
            "partitionColumns",
            "clusteringColumns"
        )
    )

for table in ["vendas_partitioned", "vendas_zorder", "vendas_liquid"]:
    print(f"\n=== {table} ===")
    table_detail(table).show(truncate=False)


=== vendas_partitioned ===
+--------------------------------------------------------+--------+-----------+------+----------------+-----------------+
|name                                                    |numFiles|sizeInBytes|format|partitionColumns|clusteringColumns|
+--------------------------------------------------------+--------+-----------+------+----------------+-----------------+
|workspace.liquid_clustering_benchmark.vendas_partitioned|20      |2141423    |delta |[categoria]     |[]               |
+--------------------------------------------------------+--------+-----------+------+----------------+-----------------+


=== vendas_zorder ===
+---------------------------------------------------+--------+-----------+------+----------------+-----------------+
|name                                               |numFiles|sizeInBytes|format|partitionColumns|clusteringColumns|
+---------------------------------------------------+--------+-----------+------+----------------+------

## 6. Harness de benchmark

O primeiro resultado é usado como **warm-up**. Depois, a query é executada várias vezes e a mediana é registrada.

In [0]:
def benchmark_sql(sql_text, repetitions=N_REPETICOES, warmup=True):
    if warmup:
        spark.sql(sql_text).count()

    times = []
    for _ in range(repetitions):
        start = time.perf_counter()
        spark.sql(sql_text).count()
        times.append(time.perf_counter() - start)

    return {
        "runs": times,
        "median_s": statistics.median(times),
        "min_s": min(times),
        "max_s": max(times)
    }

def compare_queries(query_map, repetitions=N_REPETICOES):
    rows = []
    for name, sql_text in query_map.items():
        result = benchmark_sql(sql_text, repetitions)
        rows.append((
            name,
            result["median_s"],
            result["min_s"],
            result["max_s"],
            str([round(x, 3) for x in result["runs"]])
        ))

    return spark.createDataFrame(
        rows,
        ["strategy", "median_s", "min_s", "max_s", "runs_s"]
    )

# Experimento A — Agregação

Este experimento funciona como controle: como grande parte da tabela precisa ser processada, clustering não necessariamente produzirá grande vantagem.

In [0]:
queries_agg = {
    "partitioned": '''
        SELECT categoria, AVG(valor) AS media_valor
        FROM vendas_partitioned
        GROUP BY categoria
    ''',
    "zorder": '''
        SELECT categoria, AVG(valor) AS media_valor
        FROM vendas_zorder
        GROUP BY categoria
    ''',
    "liquid": '''
        SELECT categoria, AVG(valor) AS media_valor
        FROM vendas_liquid
        GROUP BY categoria
    '''
}

agg_results = compare_queries(queries_agg)
display(agg_results)

strategy,median_s,min_s,max_s,runs_s
partitioned,0.6001341260002846,0.5772002150001754,0.8148590289997628,"[0.67, 0.592, 0.577, 0.6, 0.815]"
zorder,0.5686428469998646,0.5256142240000372,0.7934697229998164,"[0.564, 0.526, 0.793, 0.569, 0.647]"
liquid,0.6080704030000561,0.5328296669999872,0.7030439049999586,"[0.703, 0.685, 0.568, 0.533, 0.608]"


# Experimento B — Filtro seletivo / Data Skipping

Utilizamos `cliente_id` e `data_id`, alinhados às estratégias Z-ORDER e Liquid Clustering.

O objetivo é observar o comportamento de **file pruning/data skipping**.

In [0]:
TARGET_CLIENT = 12345
START_DAY = 100
END_DAY = 120

queries_filter = {
    "partitioned": f'''
        SELECT COUNT(*)
        FROM vendas_partitioned
        WHERE cliente_id = {TARGET_CLIENT}
          AND data_id BETWEEN {START_DAY} AND {END_DAY}
    ''',
    "zorder": f'''
        SELECT COUNT(*)
        FROM vendas_zorder
        WHERE cliente_id = {TARGET_CLIENT}
          AND data_id BETWEEN {START_DAY} AND {END_DAY}
    ''',
    "liquid": f'''
        SELECT COUNT(*)
        FROM vendas_liquid
        WHERE cliente_id = {TARGET_CLIENT}
          AND data_id BETWEEN {START_DAY} AND {END_DAY}
    '''
}

filter_results = compare_queries(queries_filter)
display(filter_results)

strategy,median_s,min_s,max_s,runs_s
partitioned,0.626550772000428,0.5278280249999625,0.6429551910000555,"[0.643, 0.629, 0.627, 0.54, 0.528]"
zorder,0.5213675059999332,0.5034575049999148,0.6557042089998504,"[0.527, 0.521, 0.656, 0.503, 0.516]"
liquid,0.7236632660001305,0.5183628170002521,0.7352781119998326,"[0.727, 0.518, 0.6, 0.735, 0.724]"


## 7. Plano físico do filtro

Procure principalmente:
- `PartitionFilters`
- `PushedFilters`
- `DataFilters`
- `FileScan`
- `Exchange`

O plano é a evidência usada para explicar o comportamento.

In [0]:
print("=== PARTITIONED ===")
spark.sql(queries_filter["partitioned"]).explain("formatted")

print("\n=== Z-ORDER ===")
spark.sql(queries_filter["zorder"]).explain("formatted")

print("\n=== LIQUID ===")
spark.sql(queries_filter["liquid"]).explain("formatted")

=== PARTITIONED ===
== Physical Plan ==
AdaptiveSparkPlan (10)
+- == Initial Plan ==
   PhotonResultStage (9)
   +- PhotonColumnarToRow (8)
      +- PhotonAgg (7)
         +- PhotonShuffleExchangeSource (6)
            +- PhotonShuffleMapStage (5)
               +- PhotonShuffleExchangeSink (4)
                  +- PhotonAgg (3)
                     +- PhotonProject (2)
                        +- PhotonScan parquet workspace.liquid_clustering_benchmark.vendas_partitioned (1)


(1) PhotonScan parquet workspace.liquid_clustering_benchmark.vendas_partitioned
Output [3]: [cliente_id#22000L, data_id#22002, categoria#22001]
DictionaryFilters: [(cliente_id#22000L = 12345), ((data_id#22002 >= 100) AND (data_id#22002 <= 120))]
Location: PreparedDeltaFileIndex [s3://dbstorage-prod-8v1ue/uc/3bf609dd-34df-4d95-ad8c-5f6e56d59c0a/df11ed15-862f-49d8-8215-b629f7d7b6e0/__unitystorage/catalogs/06231899-2188-483f-bb42-1992d1c00fdb/tables/75538c08-5522-4f3d-ac0e-6f58f183fad3]
ReadSchema: struct<cliente_id

# Experimento C — Join

Avaliamos um workload envolvendo vendas e clientes.

O tempo do join não deve ser interpretado isoladamente: `Exchange`, broadcast, cardinalidade e distribuição das chaves também precisam ser analisados.

In [0]:
queries_join = {
    "partitioned": '''
        SELECT c.segmento, SUM(v.valor) AS total_valor
        FROM vendas_partitioned v
        JOIN clientes c
          ON v.cliente_id = c.cliente_id
        GROUP BY c.segmento
    ''',
    "zorder": '''
        SELECT c.segmento, SUM(v.valor) AS total_valor
        FROM vendas_zorder v
        JOIN clientes c
          ON v.cliente_id = c.cliente_id
        GROUP BY c.segmento
    ''',
    "liquid": '''
        SELECT c.segmento, SUM(v.valor) AS total_valor
        FROM vendas_liquid v
        JOIN clientes c
          ON v.cliente_id = c.cliente_id
        GROUP BY c.segmento
    '''
}

join_results = compare_queries(queries_join)
display(join_results)

strategy,median_s,min_s,max_s,runs_s
partitioned,1.389815751000242,1.021228955999959,497.04442214300025,"[1.279, 1.405, 497.044, 1.39, 1.021]"
zorder,0.9483235239999885,0.8623268010001084,1.083763619000365,"[0.937, 0.948, 0.949, 0.862, 1.084]"
liquid,0.8164371790003315,0.8041245839999647,0.8712859409997691,"[0.804, 0.871, 0.813, 0.816, 0.82]"


In [0]:
print("=== JOIN - PARTITIONED ===")
spark.sql(queries_join["partitioned"]).explain("formatted")

print("\n=== JOIN - Z-ORDER ===")
spark.sql(queries_join["zorder"]).explain("formatted")

print("\n=== JOIN - LIQUID ===")
spark.sql(queries_join["liquid"]).explain("formatted")

=== JOIN - PARTITIONED ===
== Physical Plan ==
AdaptiveSparkPlan (16)
+- == Initial Plan ==
   PhotonResultStage (15)
   +- PhotonColumnarToRow (14)
      +- PhotonGroupingAgg (13)
         +- PhotonShuffleExchangeSource (12)
            +- PhotonShuffleMapStage (11)
               +- PhotonShuffleExchangeSink (10)
                  +- PhotonGroupingAgg (9)
                     +- PhotonProject (8)
                        +- PhotonBroadcastHashJoin Inner (7)
                           :- PhotonProject (2)
                           :  +- PhotonScan parquet workspace.liquid_clustering_benchmark.vendas_partitioned (1)
                           +- PhotonShuffleExchangeSource (6)
                              +- PhotonShuffleMapStage (5)
                                 +- PhotonShuffleExchangeSink (4)
                                    +- PhotonScan parquet workspace.liquid_clustering_benchmark.clientes (3)


(1) PhotonScan parquet workspace.liquid_clustering_benchmark.vendas_partitione

# Experimento D — Small Files

Criamos deliberadamente múltiplos arquivos e observamos o estado antes e depois do `OPTIMIZE`.

O objetivo é separar o efeito de **compactação de arquivos** do efeito específico de Liquid Clustering.

In [0]:
small_files_table = "vendas_small_files"

(
    vendas
    .repartition(80)
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(small_files_table)
)

print("Antes do OPTIMIZE:")
table_detail(small_files_table).show(truncate=False)

spark.sql(f"OPTIMIZE {small_files_table}")

print("Depois do OPTIMIZE:")
table_detail(small_files_table).show(truncate=False)

Antes do OPTIMIZE:
+--------------------------------------------------------+--------+-----------+------+----------------+-----------------+
|name                                                    |numFiles|sizeInBytes|format|partitionColumns|clusteringColumns|
+--------------------------------------------------------+--------+-----------+------+----------------+-----------------+
|workspace.liquid_clustering_benchmark.vendas_small_files|80      |2438601    |delta |[]              |[]               |
+--------------------------------------------------------+--------+-----------+------+----------------+-----------------+

Depois do OPTIMIZE:
+--------------------------------------------------------+--------+-----------+------+----------------+-----------------+
|name                                                    |numFiles|sizeInBytes|format|partitionColumns|clusteringColumns|
+--------------------------------------------------------+--------+-----------+------+----------------+---

# Experimento E — Z-ORDER vs Liquid Clustering

Comparação direta em um filtro alinhado às mesmas colunas de acesso.

In [0]:
query_target = '''
    SELECT COUNT(*)
    FROM {table}
    WHERE cliente_id = {client}
      AND data_id BETWEEN {start_day} AND {end_day}
'''

direct_comparison = {
    "zorder": query_target.format(
        table="vendas_zorder",
        client=TARGET_CLIENT,
        start_day=START_DAY,
        end_day=END_DAY
    ),
    "liquid": query_target.format(
        table="vendas_liquid",
        client=TARGET_CLIENT,
        start_day=START_DAY,
        end_day=END_DAY
    )
}

direct_results = compare_queries(direct_comparison)
display(direct_results)

strategy,median_s,min_s,max_s,runs_s
zorder,0.5649393609996878,0.5165204230002018,0.6800895279998258,"[0.68, 0.626, 0.565, 0.559, 0.517]"
liquid,0.5561129849997997,0.5199566260002939,0.833048613000301,"[0.52, 0.556, 0.543, 0.729, 0.833]"


# 8. Consolidação dos resultados

A tabela final pode ser exportada posteriormente para CSV e utilizada no README/GitHub.

In [0]:
def add_experiment(df, experiment):
    return df.withColumn("experiment", F.lit(experiment))

all_results = (
    add_experiment(agg_results, "aggregation")
    .unionByName(add_experiment(filter_results, "selective_filter"))
    .unionByName(add_experiment(join_results, "join"))
)

final_results = all_results.select(
    "experiment",
    "strategy",
    "median_s",
    "min_s",
    "max_s",
    "runs_s"
)

display(final_results.orderBy("experiment", "median_s"))

experiment,strategy,median_s,min_s,max_s,runs_s
aggregation,zorder,0.5686428469998646,0.5256142240000372,0.7934697229998164,"[0.564, 0.526, 0.793, 0.569, 0.647]"
aggregation,partitioned,0.6001341260002846,0.5772002150001754,0.8148590289997628,"[0.67, 0.592, 0.577, 0.6, 0.815]"
aggregation,liquid,0.6080704030000561,0.5328296669999872,0.7030439049999586,"[0.703, 0.685, 0.568, 0.533, 0.608]"
join,liquid,0.8164371790003315,0.8041245839999647,0.8712859409997691,"[0.804, 0.871, 0.813, 0.816, 0.82]"
join,zorder,0.9483235239999885,0.8623268010001084,1.083763619000365,"[0.937, 0.948, 0.949, 0.862, 1.084]"
join,partitioned,1.389815751000242,1.021228955999959,497.04442214300025,"[1.279, 1.405, 497.044, 1.39, 1.021]"
selective_filter,zorder,0.5213675059999332,0.5034575049999148,0.6557042089998504,"[0.527, 0.521, 0.656, 0.503, 0.516]"
selective_filter,partitioned,0.626550772000428,0.5278280249999625,0.6429551910000555,"[0.643, 0.629, 0.627, 0.54, 0.528]"
selective_filter,liquid,0.7236632660001305,0.5183628170002521,0.7352781119998326,"[0.727, 0.518, 0.6, 0.735, 0.724]"


# 9. Interpretação

A análise deve responder:

1. Qual estratégia apresentou menor mediana?
2. A diferença foi consistente?
3. O plano físico mudou?
4. Houve evidência de file pruning?
5. O workload realmente se beneficia da organização física?
6. Houve alteração de `Exchange`/shuffle?
7. A diferença é relevante?

**Liquid Clustering não deve ser tratado como substituto universal de partitioning ou Z-ORDER.**

A escolha depende de padrão de acesso, cardinalidade, evolução das consultas, volume, frequência de escrita e características do workload.

# 10. Limitações

Este experimento utiliza o **Databricks Free Edition**.

- O cluster é limitado.
- Os datasets são menores que workloads produtivos.
- Cache pode influenciar resultados.
- Runtime e configuração podem variar.
- O número de arquivos não representa necessariamente produção.
- Resultados de uma execução não devem ser generalizados.

Para produção, o benchmark deve ser repetido com volumes e workloads representativos.

# 11. Conclusão

O objetivo é demonstrar um processo de **Performance Engineering em Lakehouse**:

```text
Workload
   ↓
Hipótese
   ↓
Layout físico
   ↓
OPTIMIZE
   ↓
Benchmark controlado
   ↓
Métricas
   ↓
Plano físico
   ↓
Conclusão baseada em evidências
```

## Próximos experimentos

- skew de `cliente_id`
- MERGE / UPDATE
- ingestão incremental
- diferentes cardinalidades de clustering keys
- workloads multidimensionais
- concorrência
- análise de custo